#Chat Metrics

##Data Collection

In [1]:
!pip install lingua-language-detector
!pip install deep_translator

import pandas as pd
import numpy as np

from lingua import LanguageDetectorBuilder
from deep_translator import GoogleTranslator

from google.colab import drive, userdata
import os

drive.mount('/content/drive')


file_path = '/content/drive/MyDrive/ddds-cohort-21/Projects/Capstone/Data/chat_metrics_20260730_062950.xlsx'
sheets_file = pd.ExcelFile(file_path)

dfs = pd.read_excel(file_path, sheet_name=None)
df = dfs['Raw Data']
feedback = dfs['User Feedback']

Mounted at /content/drive


##Raw Data

In [2]:
#Drop excess columns
df.drop(columns=['user_id', 'correlation_id', 'hour', 'date', 'step_summaries_count', 'secondary_category'], inplace=True)

###Timestamp

In [3]:
#Convert from UTC to Mountain Time (Albuquerque)
df['timestamp'] = df['timestamp'].dt.tz_localize('UTC').dt.tz_convert('America/Denver')

#Update day of week to match local time
df['day_of_week'] = df['timestamp'].dt.day_name()

In [4]:
#Fill
df.loc[814, 'primary_category']= 'sunport_amenities'
df.loc[816, 'primary_category']= 'navigation'
df.loc[817, 'primary_category']= 'navigation'
df.loc[818, 'primary_category']= 'airline_logistics'

##Feedback

In [5]:
feedback.drop(columns=['date', 'ip_address'], inplace=True)

pd.set_option('display.max_colwidth', None)

feedback_text = feedback[feedback['feedback_text'].notna()]
feedback_text

,timestamp,star_rating,feedback_text,chat_id,session_id
147,2025-12-24 00:36:44.421,NaN,Took a bit of time but answer was great.,0f710491-a1fb-4742-ab14-ca72a3e0dd33,e3c60ae2-c602-413c-9004-9586700a8364
262,2025-12-30 12:41:51.168,NaN,I thought we fixed this outdated data,c55198b4-1deb-46e2-aa97-79b82fefc519,36d74f4f-fab1-48d2-a614-b2e92dce4a79
309,2026-01-02 11:26:57.492,NaN,You should have a live number of parking spaces available in parking garage,afdba081-c8bc-4ce3-aec4-6fb4fbdb355a,a3d925e8-31d4-43ff-9eaa-68f50bba8685
311,2026-01-02 15:47:12.870,NaN,formatting need work,578834f9-d399-4700-9ec7-3ba96640f969,4fce5ccc-683e-4d37-8b0e-c8c8e0e05bbe
761,2026-07-17 04:02:23.735,NaN,Please find a way to be able to report real-time parking spaces available. You have readouts of this when passengers arrive outside parking garage. This shouldn't be hard to do,ed78ddae-6a66-42e1-aee4-daa8d49a9f84,17b0800a-dd75-4cf1-82a7-42e45d7b5615


In [6]:
pd.reset_option('display.max_colwidth', None)

feedback.drop([147, 262, 309, 311, 761], inplace=True)
feedback.drop(columns='timestamp', inplace=True)
feedback = feedback.drop_duplicates()

##Merge

In [7]:
df = df.merge(feedback[['chat_id', 'star_rating']], on='chat_id', how='left')

##Text Cleaning

In [8]:
#Emojis
df['question'] = df['question'].replace('👍', 'Okay')

#Language Detector
detector = (LanguageDetectorBuilder.from_all_languages().build())

def detect_language(text, threshold=0.55):
  if not isinstance(text, str) or text.strip() == '':
    return 'ENGLISH'

  #Get confidence values for all languages
  confidence_values = detector.compute_language_confidence_values(text)

  #Check if highest matched meets your threshold
  if confidence_values and confidence_values[0].value > threshold:
    return confidence_values[0].language.name #Added name here

  return 'ENGLISH'

df['question_language'] = df['question'].apply(detect_language)
df['answer_language'] = df['answer'].apply(detect_language)

#Replace false positives
df['answer_language'] = df['answer_language'].replace(['LATIN', 'YORUBA', 'ESPERANTO'], 'ENGLISH')

#Translate
translator = GoogleTranslator(source='auto', target='en')

def translate_to_english(text):
  if not isinstance(text, str) or text.strip() == '':
    return text

  try:
    return translator.translate(text)
  except Exception:
    return text

df['question_en'] = df['question']
mask = df['question_language'] != 'ENGLISH'
df.loc[mask, 'question_en'] = (df.loc[mask, 'question'].apply(translate_to_english))

df['answer_en'] = df['answer']
mask = df['answer_language'] != 'ENGLISH'
df.loc[mask, 'answer_en'] = (df.loc[mask, 'answer'].apply(translate_to_english))

df.drop(columns=['chat_id', 'question', 'answer', 'question_language', 'answer_language'], inplace=True)

##Agggregate

In [9]:
#Function to round star rating avg to neaarest half dec
def round_to_half(x):
  return round(x * 2) / 2

df_clean = df.groupby('session_id').agg(
    timestamp = ('timestamp', 'first'),
    day_of_week = ('day_of_week', lambda x: x.mode().loc[0] if not x.mode().empty else None),
    processing_time_seconds = ('processing_time_seconds', 'sum'),
    total_tokens = ('total_tokens', 'sum'),
    input_tokens = ('input_tokens', 'sum'),
    output_tokens = ('output_tokens', 'sum'),
    model_calls = ('model_calls', 'sum'),
    tool_calls_count = ('tool_calls_count', 'sum'),
    question_length = ('question_length', 'sum'),
    answer_length = ('answer_length', 'sum'),
    primary_category = ('primary_category', lambda x: ', '.join(x.dropna().unique())),
    selected_agent = ('selected_agent', lambda x: ', '.join(x.dropna().unique())),
    has_geolocation = ('has_geolocation', 'any'),
    star_rating = ('star_rating', lambda x: round_to_half(x.mean()) if pd.notna(x.mean()) else np.nan),
    question_en = ('question_en', lambda x: ' '.join(x.dropna())),
    answer_en = ('answer_en', lambda x: ' '.join(x.dropna()))
).reset_index()

#Drop identifier
df_clean.drop(columns='session_id', inplace=True)

#Multi-Label Binary Encoding
#Primary Categories
categories = [
  'greetings',
  'sunport_amenities',
  'navigation',
  'airline_logistics',
  'general_info'
]

for category in categories:
  df_clean[f'primary_category_{category}'] = (
    df_clean['primary_category']
    .str.contains(category, regex=False, na=False)
    .astype('int8')
  )

df_clean.drop(columns='primary_category', inplace=True)

#Selected Agent
agents = [
  'reporter',
  'planner',
  'location',
  'location_fallback_to_reporter',
  'broad_search_synthesis',
  'broad_search_passthrough'
]

for agent in agents:
  df_clean[f'selected_agent{agent}'] = (
    df_clean['selected_agent']
    .str.contains(agent, regex=False, na=False)
    .astype('int8')
  )

df_clean.drop(columns='selected_agent', inplace=True)

#Datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])

df_clean['year'] = df_clean['timestamp'].dt.year
df_clean['month'] = df_clean['timestamp'].dt.month
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['hour'] = df_clean['timestamp'].dt.hour

df_clean.drop(columns='timestamp', inplace=True)

#Cyclical Encoding
#Days of the Week
days = {
  'Monday':0,
  'Tuesday':1,
  'Wednesday':2,
  'Thursday':3,
  'Friday':4,
  'Saturday':5,
  'Sunday':6
}

df_clean['day_num'] = df_clean['day_of_week'].map(days)

df_clean['day_sin'] = np.sin(2 * np.pi * df_clean['day_num'] / 7)
df_clean['day_cos'] = np.cos(2 * np.pi * df_clean['day_num'] / 7)

#Hour
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

#Month
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.sin(2 * np.pi * df_clean['month'] / 12)

#Save as a Parquet

In [10]:
parquet_file = 'chat_text.parquet'
df.to_parquet(parquet_file, index=False)
!ls -l --si {parquet_file}

os.environ["HF_TOKEN"] = userdata.get('hf_cs_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]

hf_org = "ddds-Capstone"
os.environ["HF_ORG"] = hf_org

hf_repo = "Datasets"
os.environ["HF_REPO"] = hf_repo

!hf auth login --token $HF_TOKEN

-rw-r--r-- 1 root root 3.0M Aug 19 02:00 chat_text.parquet
Hint: A new version of huggingface_hub (1.28.0) is available! You are using version 1.27.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Capstone Token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ORG}/${HF_REPO} \
  chat_text.parquet

In [12]:
hf_url = f"https://huggingface.co/datasets/{hf_org}/{hf_repo}/resolve/main/chat_metrics.parquet"
hf_url

'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet'